# Item 5: 1,000 photographic FLUX.1 faces from SFHQ-T2I, cropped exactly like the SD3 set

Builds `item5_faces_v2`, a self-contained test set for faculty item 5:
- `real_ffhq` and `sd3`, copied unchanged from `item5-faces`.
- **`flux_dev`** (500) and **`flux_pro`** (500): new FLUX.1 faces from **SFHQ-T2I** (Kaggle, selfishgene, MIT licence).
  - These are curated single-face images whose prompts vary ethnicity, age, pose, lighting and background.
  - The model, prompt and seed are recorded per image.
  - Sampling is random (seed 42); prompts that ask for paintings, renders or other styles are dropped.
- `flux_stylised`: an earlier FLUX set (gaunernst/flux-dev-portrait), kept only as a comparison. It is stylised and skewed.

Every new face goes through the **same crop as the SD3 set**: MTCNN at 160 px, margin 16, exactly one face with probability 0.95 or higher, a box of at least 96 px, then saved as JPEG at quality 90. The notebook ends with a check:
- image counts
- prompt demographics
- a shortcut audit: can simple statistics separate each source from the real faces, with and without colour normalisation?
- an example sheet

**Inputs:**
1. **SFHQ-T2I** (+ Add Input, Datasets, search `SFHQ-T2I`, owner selfishgene, about 24.5 GB).
2. `item5-faces`.

**Settings:** Accelerator None (CPU), so no GPU quota is used; Internet ON (to install facenet-pytorch).

**Run:** Save & Run All (Commit). Then attach **this notebook's output** (not `item5-faces`) to `10_new_generator_scoring.ipynb`.

## Cell 1: setup

In [ ]:
!pip install -q --no-deps facenet-pytorch   # --no-deps keeps Kaggle's torch untouched
import os, re, json, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from facenet_pytorch import MTCNN

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp')
print('torch', torch.__version__, '| device', DEV)

## Cell 2: configuration and inputs

In [ ]:
# ===================== CONFIG AND INPUTS =====================
N_KEEP = {'dev': 500, 'pro': 500}   # final faces per FLUX.1 model
SEED = 42
STYLE = r'paint|illustrat|cartoon|anime|\b3d\b|render|digital art|sketch|watercolou?r|oil on|cgi|drawing|comic|pixar|sculpt|statue|fantasy art|concept art'
OUT = Path('/kaggle/working/item5_faces_v2')
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)


def find(name, top='/kaggle/input', max_depth=8):
    """Folders holding `name`, without listing folders full of images."""
    hits = []
    for dp, dns, fns in os.walk(top):
        if name in fns:
            hits.append(Path(dp) / name)
        if dp.count(os.sep) >= max_depth or sum(f.lower().endswith(IMG_EXT) for f in fns) > 50:
            dns[:] = []
    return sorted(hits)


sfhq_csv = find('SFHQ_T2I_dataset.csv')
assert sfhq_csv, 'Attach SFHQ-T2I (owner selfishgene)'
SFHQ_CSV = sfhq_csv[0]
item5 = [p for p in find('manifest.csv') if {'path', 'source', 'label'} <= set(pd.read_csv(p, nrows=1).columns)]
assert len(item5) == 1, f'Attach item5-faces exactly once; found {item5}'
ITEM5 = item5[0].parent
print('SFHQ-T2I csv:', SFHQ_CSV, '| item5-faces:', ITEM5)

t = time.time()
FILES = {}   # SFHQ image file name -> path (the folder layout is not documented, so map by name)
for dp, dns, fns in os.walk(SFHQ_CSV.parent):
    for f in fns:
        if f.lower().endswith(IMG_EXT):
            FILES[f] = os.path.join(dp, f)
print(f'{len(FILES):,} SFHQ-T2I image files indexed in {time.time() - t:.0f} s')

## Cell 3: filter to photographic FLUX.1 dev and pro, random order

In [ ]:
# ===================== FILTER AND SAMPLE =====================
df = pd.read_csv(SFHQ_CSV)
print('columns:', df.columns.tolist())
print(df.model_used.value_counts().to_string())
models = {k: [m for m in df.model_used.unique() if 'flux' in str(m).lower() and k in str(m).lower()] for k in N_KEEP}
assert all(len(v) == 1 for v in models.values()), f'Check the FLUX model names: {models}'
models = {k: v[0] for k, v in models.items()}
print('using', models)

df['path'] = df.image_filename.map(FILES)
CANDS = {}
for k, m in models.items():
    d = df[df.model_used.eq(m)]
    photo = d[~d.text_prompt.str.contains(STYLE, case=False, regex=True, na=False)]
    found = photo.dropna(subset=['path'])
    print(f'{m}: {len(d):,} images | {len(photo):,} after the style filter | {len(found):,} with a file')
    CANDS[k] = found.sample(frac=1, random_state=SEED).reset_index(drop=True)   # random order, fixed by the seed

## Cell 4: crop, identical to the SD3 set; copy the reals and SD3

In [ ]:
# ===================== CROP: identical to the SD3 set (Step 4 of the item 5 instructions) =====================
mtcnn = MTCNN(image_size=160, margin=16, post_process=False, device=DEV)


def crop_face(src, dst, min_box=96, min_prob=0.95):
    img = Image.open(src).convert('RGB')
    boxes, probs = mtcnn.detect(img)
    if boxes is None or len(boxes) != 1 or probs[0] < min_prob:
        return False                       # no face, or more than one: skip the image
    x1, y1, x2, y2 = boxes[0]
    if min(x2 - x1, y2 - y1) < min_box:
        return False                       # face too small to judge
    face = mtcnn.extract(img, boxes, None) # 160 x 160, raw pixels
    Image.fromarray(face.permute(1, 2, 0).clamp(0, 255).byte().numpy()).save(dst, quality=90)
    return True


rows, tried = [], {}
for k, cands in CANDS.items():
    src_name = f'flux_{k}'
    (OUT / src_name).mkdir()
    kept = n = 0
    for r in cands.itertuples():
        if kept == N_KEEP[k]:
            break
        n += 1
        dst = OUT / src_name / f'{src_name}_{kept:05d}.jpg'
        if crop_face(r.path, dst):
            rows.append(dict(path=f'{src_name}/{dst.name}', source=src_name, label=1, video_id=dst.name,
                             prompt=r.text_prompt, model_used=r.model_used, sfhq_file=r.image_filename,
                             configs=getattr(r, 'configs', '')))
            kept += 1
        if n % 200 == 0:
            print(f'  {src_name}: tried {n}, kept {kept}', flush=True)
    tried[src_name] = n
    assert kept == N_KEEP[k], f'{src_name}: only {kept} faces after {n} candidates'
    print(f'{src_name}: kept {kept} of {n} tried ({n - kept} dropped at the crop)')

old = pd.read_csv(ITEM5 / 'manifest.csv', dtype=str)   # copy the reals, SD3, and the old stylised FLUX set unchanged
old['source'] = old.source.replace({'flux': 'flux_stylised'})
for r in old.itertuples():
    rel = Path(r.path)
    src = ITEM5 / rel if (ITEM5 / rel).is_file() else ITEM5 / Path(*rel.parts[-2:])
    dst = OUT / r.source / src.name
    dst.parent.mkdir(exist_ok=True)
    shutil.copy2(src, dst)
    rows.append(dict(path=f'{r.source}/{src.name}', source=r.source, label=int(r.label), video_id=f'{r.source}_{src.name}',
                     prompt=r.prompt if isinstance(r.prompt, str) else '', model_used='', sfhq_file='', configs=''))
MAN = pd.DataFrame(rows)
MAN.to_csv(OUT / 'manifest.csv', index=False)
print(MAN.groupby(['source', 'label']).size().to_string())

## Cell 5: check, then download

In [ ]:
# ===================== CHECK: demographics, shortcut audit, examples =====================
# Path B (a folder packaged by hand): run Cell 1, then uncomment these two lines and run this cell alone.
# OUT = Path('/kaggle/working/item5_faces_v2')
# MAN = pd.read_csv(OUT / 'manifest.csv', dtype={'path': str, 'source': str, 'prompt': str}).fillna({'prompt': ''})
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from IPython.display import FileLink, display

KEYS = dict(woman=r'\b(?:woman|women|girl|female|lady)\b', man=r'\b(?:man|men|boy|male|gentleman|guy)\b',
            old=r'\b(?:old|elderly|older|aged|senior|grandfather|grandmother)\b', young=r'\b(?:young|teen|teenage|child|kid|youthful)\b')
demo = MAN[MAN.prompt.fillna('').str.len() > 0].groupby('source').prompt.apply(
    lambda s: pd.Series({k: s.str.contains(v, case=False, regex=True).mean() for k, v in KEYS.items()})).unstack()
print('Share of prompts mentioning each word:')
display(demo.round(3))

files = [OUT / p for p in MAN.path]
arrs = {}
for p in files:
    with Image.open(p) as im:
        arrs[p] = np.asarray(im.convert('RGB').resize((160, 160)), dtype=np.float32)
real = [arrs[OUT / p] for p, s in zip(MAN.path, MAN.source) if s == 'real_ffhq']
REF_MEAN = np.mean([a.mean((0, 1)) for a in real], 0)
REF_STD = np.mean([a.std((0, 1)) for a in real], 0)


def stats(a, norm):   # the same 8 statistics and colour normalisation as 10_new_generator_scoring.ipynb
    if norm:
        a = np.clip((a - a.mean((0, 1))) / (a.std((0, 1)) + 1e-6) * REF_STD + REF_MEAN, 0, 255)
    a = a / 255
    g = a.mean(2)
    lap = (np.roll(g, 1, 0) + np.roll(g, -1, 0) + np.roll(g, 1, 1) + np.roll(g, -1, 1) - 4 * g)[1:-1, 1:-1]
    f = np.abs(np.fft.fftshift(np.fft.fft2(g)))
    yy, xx = np.ogrid[:160, :160]
    return [*a.mean((0, 1)), *a.std((0, 1)), lap.var(), f[np.hypot(yy - 80, xx - 80) > 40].sum() / f.sum()]


audit = []
for norm in (False, True):
    X = np.array([stats(arrs[p], norm) for p in files])
    for s in sorted(set(MAN.source) - {'real_ffhq'}):
        k = MAN.source.isin(['real_ffhq', s]).values
        y = MAN.label.values[k].astype(int)
        pr = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X[k], y,
                               cv=StratifiedKFold(5, shuffle=True, random_state=42), method='predict_proba')[:, 1]
        audit.append(dict(source=s, colour_norm=norm, simple_stats_auc=roc_auc_score(y, pr)))
AUDIT = pd.DataFrame(audit)
AUDIT.to_csv(OUT / 'shortcut_audit.csv', index=False)
print('Simple-statistics AUC against real_ffhq (lower = harder shortcut, 0.5 = none):')
display(AUDIT.pivot(index='source', columns='colour_norm', values='simple_stats_auc').round(3))

srcs = sorted(MAN.source.unique())
sheet = Image.new('RGB', (16 * 96, len(srcs) * 96), 'white')
for r, s in enumerate(srcs):
    pick = MAN[MAN.source.eq(s)].sample(min(16, (MAN.source == s).sum()), random_state=42).path
    for c, p in enumerate(pick):
        sheet.paste(Image.fromarray(arrs[OUT / p].astype(np.uint8)).resize((96, 96)), (c * 96, r * 96))
sheet.save(OUT / 'examples.png')
print('examples.png rows, top to bottom:', srcs)
display(sheet)

g = globals().get   # Path B runs this cell alone, so the Path A settings may be missing
meta = dict(seed=g('SEED', 42), kept=g('N_KEEP'), tried=g('tried'), models=g('models'), style_filter=g('STYLE'),
            sfhq_csv=str(g('SFHQ_CSV', '')), item5=str(g('ITEM5', '')), counts=MAN.groupby('source').size().to_dict(), crop='MTCNN 160 px, margin 16, one face, prob >= 0.95, box >= 96 px, JPEG q90',
            licence='SFHQ-T2I: MIT (Kaggle and GitHub). FLUX.1 [dev] outputs: BFL non-commercial licence (research use).')
(OUT / 'build_metadata.json').write_text(json.dumps(meta, indent=2))
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=OUT.name)
display(FileLink(os.path.relpath(archive, os.getcwd())))